# Model Evaluation

Compare all three sentiment models (TF-IDF, LSTM, BERT) on the test set.

## Imports

In [ ]:
import os
import sys
import pandas as pd
import pickle
import torch
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Load Test Data

In [ ]:
test_df = pd.read_csv('data/test.csv').dropna()
# Sample for faster evaluation — remove this for full evaluation
test_df = test_df.sample(500, random_state=42)
print(f"Test set size: {len(test_df)}")
test_df.head()

## 1. Evaluate TF-IDF + Logistic Regression

In [ ]:
print("--- Evaluating TF-IDF + LR ---")
model_dir = 'models/tfidf_lr'

if os.path.exists(model_dir):
    with open(os.path.join(model_dir, 'tfidf_vectorizer.pkl'), 'rb') as f:
        vectorizer = pickle.load(f)
    with open(os.path.join(model_dir, 'lr_model.pkl'), 'rb') as f:
        lr_model = pickle.load(f)
    
    X_test = test_df['review'].values
    y_test = (test_df['sentiment'] == 'positive').astype(int).values
    
    X_test_vec = vectorizer.transform(X_test)
    tfidf_preds = lr_model.predict(X_test_vec)
    
    print(f"Accuracy: {accuracy_score(y_test, tfidf_preds):.4f}")
    print(classification_report(y_test, tfidf_preds, target_names=['negative', 'positive']))
else:
    print("TF-IDF model not found. Train it first using 02_tfidf_model.ipynb")

## 2. Evaluate LSTM

In [ ]:
print("--- Evaluating LSTM ---")

## 3. Evaluate BERT

In [ ]:
print("--- Evaluating BERT ---")
model_dir = 'models/bert_sentiment'

# Also check for 'models/bert/best_model' (legacy path)
if not os.path.exists(model_dir):
    model_dir = 'models/bert/best_model'

if os.path.exists(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    bert_model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    bert_model.to(device)
    bert_model.eval()
    
    X_test = test_df['review'].values
    y_test = (test_df['sentiment'] == 'positive').astype(int).values
    
    bert_preds = []
    batch_size = 16
    with torch.no_grad():
        for i in range(0, len(X_test), batch_size):
            batch_texts = X_test[i:i+batch_size].tolist()
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
            outputs = bert_model(**inputs)
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
            bert_preds.extend(batch_preds)
    
    print(f"Accuracy: {accuracy_score(y_test, bert_preds):.4f}")
    print(classification_report(y_test, bert_preds, target_names=['negative', 'positive']))
else:
    print("BERT model not found. Train first using 04_bert_model.ipynb")